# Binary neural-network forward pass

**Status:** `EDUCATIONAL` · completed reference exercise · not research evidence

This original, self-contained exercise demonstrates a `400 → 25 → 15 → 1` sigmoid network, a dense layer implemented in NumPy, and a vectorized equivalence check. No source course notebook or personal course record is redistributed.

The executable checks below validate this educational implementation only. They do not validate KakeyaLogic, Excellence Engine V4, AI-alignment, or theorem-facing claims.

## Learning goals

1. Keep every layer's input/output dimensions explicit.
2. Compute one dense layer as `g(a_in @ W + b)`.
3. Verify that loop and vectorized implementations agree numerically.
4. Separate an executable educational receipt from any research-status claim.

In [ ]:
import numpy as np

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-z))

## Exercise 1 — specify the model

The input is a flattened `20 × 20` grayscale image (`400` features). The output is one sigmoid probability for binary classification.

In [ ]:
ARCHITECTURE = (
    (400, 25, "sigmoid"),
    (25, 15, "sigmoid"),
    (15, 1, "sigmoid"),
)

def build_keras_model():
    """Build the same architecture when TensorFlow is installed."""
    import tensorflow as tf

    return tf.keras.Sequential(
        [
            tf.keras.Input(shape=(400,)),
            tf.keras.layers.Dense(25, activation="sigmoid"),
            tf.keras.layers.Dense(15, activation="sigmoid"),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ],
        name="binary_digit_model",
    )

assert ARCHITECTURE == ((400, 25, "sigmoid"), (25, 15, "sigmoid"), (15, 1, "sigmoid"))

## Exercise 2 — implement a dense layer with a loop

In [ ]:
def my_dense(a_in, W, b, activation):
    """Compute one dense layer for a single input vector."""
    a_in = np.asarray(a_in, dtype=float)
    W = np.asarray(W, dtype=float)
    b = np.asarray(b, dtype=float)

    if a_in.ndim != 1 or W.ndim != 2 or b.ndim != 1:
        raise ValueError("Expected a_in (n,), W (n,j), and b (j,)")
    if W.shape != (a_in.size, b.size):
        raise ValueError("Layer dimensions do not align")

    a_out = np.zeros(W.shape[1], dtype=float)
    for j in range(W.shape[1]):
        z = np.dot(a_in, W[:, j]) + b[j]
        a_out[j] = activation(z)
    return a_out

In [ ]:
x_test = 0.1 * np.arange(1, 3).reshape(2,)
W_test = 0.1 * np.arange(1, 7).reshape(2, 3)
b_test = 0.1 * np.arange(1, 4).reshape(3,)
expected = np.array([0.54735762, 0.57932425, 0.61063923])
actual = my_dense(x_test, W_test, b_test, sigmoid)
np.testing.assert_allclose(actual, expected, rtol=0, atol=1e-8)
print(actual)

## Exercise 3 — vectorize and cross-check

In [ ]:
def my_dense_vectorized(A_in, W, b, activation):
    """Compute a dense layer for a batch shaped (m,n)."""
    return activation(np.matmul(A_in, W) + b)

rng = np.random.default_rng(7)
A_batch = rng.normal(size=(8, 4))
W_batch = rng.normal(size=(4, 3))
b_batch = rng.normal(size=(3,))
loop_result = np.vstack([my_dense(row, W_batch, b_batch, sigmoid) for row in A_batch])
vector_result = my_dense_vectorized(A_batch, W_batch, b_batch, sigmoid)
np.testing.assert_allclose(loop_result, vector_result, rtol=0, atol=1e-12)
print("maximum loop/vector difference:", float(np.max(np.abs(loop_result - vector_result))))

## Receipt and boundary

**Pass condition:** the architecture assertion, known-value dense-layer check, and loop/vector equivalence check all complete without error.

**Failure condition:** any dimension mismatch, numerical mismatch, or unavailable optional TensorFlow dependency is reported directly.

**Research boundary:** this notebook demonstrates implementation discipline and numerical equivalence only. It supplies no evidence for geometric, spectral, alignment, Coleman-conjecture, or Riemann-hypothesis claims.